In [ ]:
import pandas as pd
import time

In [ ]:
!pip install selenium -q

In [ ]:
# 라이브러리 설치
!pip install undetected-chromedriver -q

#라이브러리 불러오기
import undetected_chromedriver as uc

In [ ]:
# 크롤링에 필요한 라이브러리 불러오기
# 웹을 탐색할 라이브러리
from selenium import webdriver as wb

# 선택자를 지정할수 있게 해주는 라이브러리
from selenium.webdriver.common.by import By

# 키보드를 제어하는 라이브러리
from selenium.webdriver.common.keys import Keys

# 시간을 제어할 라이브러리
import time

In [ ]:
# 사용법은 webdriver 동일
# 경우에 따라 version 표시는 필요
# uc 사용할때, version_main 명시 필요
# uc 사용할때, 크롬 버전과 크롬driver가 서로 맞지 않을때 session error발생
# 대처방법
# 1. version_main 활용해서, 버전을 맞춰주는 방법
# 2. !pip install --upgrade undetected-chromedriver
driver = uc.Chrome(version_main=146)
driver.get("https://glowpick.co.kr/ranking/category/2/6")

In [ ]:
# 상품 클릭을 위한 태그 수집
item = driver.find_elements(By.XPATH,"/html/body/div[3]/main/section/div[2]/ul/li/a")
print(len(item))

#/html/body/div[3]/main/section/div[2]/ul/li[1]/a/div[1]
items = [el.get_attribute("href") for el in item]
print(items)

In [ ]:
import random

all_data = []

for item_url in items:
    driver.get(item_url)
    time.sleep(random.uniform(2, 4))
    
    # 제품명 추출
    product_name = driver.find_element(By.XPATH, "/html/body/div[2]/main/section[1]/div/article/div[2]/h1").text
    print(f"제품명: {product_name}")
    
    # 이미지 URL 추출
    img_url = driver.find_element(By.XPATH, "/html/body/div[2]/main/section[1]/div/div[1]/img").get_attribute("src")
    
    # 리뷰 수집
    page_num = 2

    while True:
        reviews = driver.find_elements(By.XPATH, "/html/body/div[2]/main/section[3]/div[1]/div[1]/ul/li/a/div/div/div[3]")
        ratings = driver.find_elements(By.XPATH, "/html/body/div[2]/main/section[3]/div[1]/div[1]/ul/li/a/div/div/div[2]/div/span")
        
        for r, rate in zip(reviews, ratings):  # 리뷰랑 별점 같이 묶기
            all_data.append({
                "제품명": product_name,
                "이미지URL": img_url,
                "별점": rate.text.strip(),
                "리뷰": r.text.strip()
            })
        print(f"  {page_num-1}페이지 수집 완료, 현재까지 {len(all_data)}개")
        
        try:
            next_btn = driver.find_element(By.XPATH, f"/html/body/div[2]/main/section[3]/div[1]/div[2]/div/ul/li[{page_num}]")
            next_btn.click()
            time.sleep(random.uniform(2, 4))
            page_num += 1
        except:
            print(f"  [{product_name}] 마지막 페이지 도달")
            time.sleep(random.uniform(1, 2))
            break

print(f"\n총 수집 리뷰 수: {len(all_data)}개")
df = pd.DataFrame(all_data)
print(df.head())
df.to_csv("glowpick_face_oil_reviews.csv", index=False, encoding="utf-8-sig")
print("저장 완료!")
